In [2]:
## Necessary imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.stats.multitest import multipletests

In [4]:
hier_clustering_cell_perc_fname = '../cell_percentage_hier_corr 10Hz.csv'

# Load hierarchical clustering cell percentage data
# - Each row = one trial
# - Each column = one cluster category (e.g., Cluster 0, Cluster 1, ...)
cell_percentage_hier = pd.read_csv(hier_clustering_cell_perc_fname)

# Preview the bottom rows to inspect structure and completeness
cell_percentage_hier.tail()

FileNotFoundError: [Errno 2] No such file or directory: '../cell_percentage_hier_corr 10Hz.csv'

In [ ]:
# Extract cluster-1 cell percentages for each stimulated animal
# NaN values (empty clusters) are removed
cluster1_percentage = [
cell_percentage_hier[k].dropna().values
for k in ['G7 (#1)', 'G8 (#1)', 'G9 (#1)', 'G28 (#1)', 'G29 (#1)', 'G37 (#1)']
]

# Extract cluster-2 cell percentages for each stimulated animal
# NaN values (empty clusters) are removed
cluster2_percentage = [
cell_percentage_hier[k].dropna().values
for k in ['G7 (#2)', 'G8 (#2)', 'G9 (#2)', 'G28 (#2)', 'G29 (#2)', 'G37 (#2)']
]

flat_cluster1stim = np.concatenate(cluster1_percentage)
flat_cluster2stim = np.concatenate(cluster2_percentage)

# Extract cluster percentages for each restraint stress animal 
flat_cluster1stress = cell_percentage_hier['Stress (#1)'].dropna().values
flat_cluster2stress = cell_percentage_hier['Stress (#2)'].dropna().values

# Animal labels corresponding to the flattened cluster entries
stim_animal = [
'G7', 'G7',
'G8', 'G8', 'G8', 'G8',
'G9', 'G9', 'G9', 'G9',
'G28', 'G28', 'G28',
'G29', 'G29', 'G29',
'G37', 'G37', 'G37', 'G37'
]

stress_animal = ['G7', 'G8', 'G9', 'G28', 'G29', 'G37', 'G37']

def build_df_two_clusters(animal_list, y1, y2, condition):
    y1 = np.asarray(y1, dtype=float)
    y2 = np.asarray(y2, dtype=float)
    animal_list = np.asarray(animal_list)

    if len(animal_list) != len(y1) or len(animal_list) != len(y2):
        raise ValueError(
            f"Length mismatch: animals={len(animal_list)}, y1={len(y1)}, y2={len(y2)}"
        )

    df = pd.DataFrame({
        "Animal_id": animal_list,
        "y1": y1,   # cluster 1 size
        "y2": y2    # cluster 2 size
    })

    df["condition"] = condition
    df["repeat"] = df.groupby("Animal_id").cumcount() + 1
    return df[["Animal_id", "condition", "repeat", "y1", "y2"]]


# --- Build condition-specific dataframes ---
df_stim = build_df_two_clusters(stim_animal, flat_cluster1stim, flat_cluster2stim, "stim")
df_stress = build_df_two_clusters(stress_animal, flat_cluster1stress, flat_cluster2stress, "stress")

# --- Merge into one dataframe ---
df = pd.concat([df_stim, df_stress], ignore_index=True)
df = df.sort_values(["condition", "Animal_id", "repeat"]).reset_index(drop=True)

print(df)

In [ ]:
# df has columns: Animal_id, condition, repeat, y1, y2
df_long = (
    df[["Animal_id", "condition", "repeat", "y1", "y2"]]
    .melt(
        id_vars=["Animal_id", "condition", "repeat"],
        value_vars=["y1", "y2"],
        var_name="cluster",
        value_name="y"
    )
)

# clean cluster labels
df_long["cluster"] = df_long["cluster"].str.strip().map({"y1": "cluster1", "y2": "cluster2"})

# drop any rows where y is missing (should usually be none)
df_long = df_long.dropna(subset=["y"]).reset_index(drop=True)

print(df_long.head(10))
print(df_long.columns)

In [ ]:
# Fit mixed-effects model with interaction between cluster and condition
m_nested = smf.mixedlm(
    "y ~ cluster * condition",
    df_long,
    groups=df_long["Animal_id"],
).fit(reml=False, method="nm", maxiter=2000, disp=True)

print(m_nested.summary())

var_fixed = np.var(m_nested.fittedvalues)
var_random = m_nested.cov_re.iloc[0, 0]
var_resid = m_nested.scale

R2_marginal = var_fixed / (var_fixed + var_random + var_resid)
print(R2_marginal)

pvals_all_clusters =m_nested.pvalues.get("cluster[T.cluster2]")
print(pvals_all_clusters)

In [ ]:
# Subset to stimulation only
df_stim_only = df_long[df_long["condition"] == "stim"].copy()
# Fit mixed-effects model: cluster effect within stim
m_stim = smf.mixedlm(
    "y ~ cluster",
    df_stim_only,
    groups=df_stim_only["Animal_id"]
).fit( method="nm", maxiter=2000, disp=True)

print(m_stim.summary())

var_fixed = np.var(m_stim.fittedvalues)
var_random = m_stim.cov_re.iloc[0, 0]
var_resid = m_stim.scale

R2_marginal = var_fixed / (var_fixed + var_random + var_resid)
print(R2_marginal)

pvals_stim_clusters =m_stim.pvalues.get("cluster[T.cluster2]")
print(pvals_stim_clusters)

In [ ]:
# Subset to stimulation only
df_stress_only = df_long[df_long["condition"] == "stress"].copy()
# Fit mixed-effects model: cluster effect within stim
m_stress = smf.mixedlm(
    "y ~ cluster",
    df_stress_only,
    groups=df_stress_only["Animal_id"]
).fit( method="nm", maxiter=2000, disp=True)

print(m_stress.summary())

var_fixed = np.var(m_stress.fittedvalues)
var_random = m_stress.cov_re.iloc[0, 0]
var_resid = m_stress.scale

R2_marginal = var_fixed / (var_fixed + var_random + var_resid)
print(R2_marginal)

pvals_stress_clusters =m_stress.pvalues.get("cluster[T.cluster2]")
print(pvals_stress_clusters)